# sqrt-eps-stabilize — worked example 1: Implement the Adam denominator with eps inside the sqrt

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sqrt-eps-stabilize`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

In Adam and RMSprop, the parameter update is scaled by `g / (sqrt(v) + eps)`. However, placing eps outside the sqrt only prevents exact divide-by-zero; it does not bound the gradient of sqrt(v) near v=0. The correct stabilization is `g / sqrt(v + eps)`, which ensures the denominator derivative remains bounded everywhere. This is the canonical form used in all modern implementations.

## Worked solution

**Step 1 — Create a vector of second-moment estimates.** We include a zero entry (dead unit) and several small values to stress-test the stabilization.

**Step 2 — Show what happens without eps.** `g / sqrt(v)` produces NaN for the zero entry and very large values for near-zero entries.

**Step 3 — Apply eps inside the sqrt.** `g / sqrt(v + eps)` gives finite, reasonable values for every entry.

**Step 4 — Compare the two.** Print both to show the contrast. The stabilized version is smooth even at v=0.

In [ ]:
import torch as t

t.manual_seed(0)

# Simulated gradient and second-moment estimates (one unit dead, others vary)
g = t.tensor([1.0, 1.0, 1.0, 1.0, 1.0])
v = t.tensor([0.0, 1e-10, 1e-5, 0.01, 1.0])
eps = 1e-8

# Unstabilized (outside sqrt)
scale_bad = g / (t.sqrt(v) + eps)

# Stabilized (inside sqrt)
scale_good = g / t.sqrt(v + eps)

print('v:           ', v.tolist())
print('Unstabilized:', [f'{x:.2e}' if not t.isnan(t.tensor(x)) else 'NaN' for x in scale_bad.tolist()])
print('Stabilized:  ', [f'{x:.2e}' for x in scale_good.tolist()])
print('Stabilized finite:', t.isfinite(scale_good).all().item())
print('At v=0, unstabilized is finite:', t.isfinite(scale_bad[0]).item())  # may be inf/nan
print('At v=0, stabilized =', scale_good[0].item(), '(bounded by 1/sqrt(eps))')